# This notebook removes large-residual arrival-time data using linear regression.

In [ ]:
# Import TomoATT data-processing utilities
import sys
sys.path.append('../utils')
import functions_for_data as ffd

# 1. Read the Arrival-Time Data File

In [ ]:
# Read data
fname = "output_data/step1_src_rec_indom.dat"
[ev_info_obs, st_info_obs] = ffd.read_src_rec_file(fname)

# Data plot (optional); set fname = None to skip saving the figure.
ffd.fig_ev_st_distribution_dep(ev_info_obs, st_info_obs, fname = None)

# 2. Remove preliminary outliers so extreme values do not bias the regression line.

In [ ]:
# Filtering rule: discard data outside the bounds defined by slope * distance + intercept + up/down.
slope = 0.16    # slope
intercept = 0   # intercept
up = 10         # upper bound
down = -10      # lower bound

# Hypocentral-distance range between earthquakes and stations
dis_min = 0 
dis_max = 500

# Remove data with residuals above the threshold or distances outside the selected range.
ev_remove_outlier = ffd.fig_data_plot_remove_outliers(ev_info_obs,st_info_obs,slope,intercept,up,down,dis_min,dis_max, fname = "figs/step2_data_remove_outlier.png")

# 3. Remove Data with Epicentral Distances Larger than 100 km

The distance-time plot shows that when epicentral distance is large enough, two branches appear. This is especially clear near 300 km.

The lower branch has smaller arrival times and corresponds to Pn waves, while the upper branch has larger arrival times and corresponds to Pg waves.

TomoATT computes first arrivals. At smaller epicentral distances, Pg is usually the first-arrival phase, whereas at larger distances Pn becomes the first arrival.

The critical distance separating the two phases depends on velocity structure, Moho depth, source depth, and other factors, so it is usually difficult to determine robustly.

In general, when epicentral distance is smaller than 100 km, most first arrivals are Pg waves. To avoid data contamination, this example removes all data with epicentral distance larger than 100 km and also removes explicit Pn phases, so the retained data are first-arrival P-wave times.

In [ ]:
# Remove data whose epicentral distances fall within [epi_dis1, epi_dis2].
epi_dis1 = 100      
epi_dis2 = 1000000
ev_limit_dis = ffd.limit_epi_dis(ev_remove_outlier, st_info_obs, epi_dis1, epi_dis2)

# Within 100 km epicentral distance, Pn is usually not the first arrival, so remove Pn and keep P, Pg, and Pb.
phase_list = ["P","Pg","Pb"]
ev_limit_dis_phase = ffd.limit_data_phase(ev_limit_dis,phase_list)

# Data plot (optional)
phase_list = ["P","Pg","Pb"]   # phases and plotting colors
color_list = ["b","r","g"]

dis_min = 0     # hypocentral-distance plotting range
dis_max = 110

# See the English README for details.
ffd.fig_data_plot_phase(ev_limit_dis_phase,st_info_obs,phase_list,color_list,dis_min,dis_max, fname = "figs/step2_data_limit_distance.png")

# 4. Perform Linear Regression on the Arrival-Time Data
### Keep arrival times whose residuals are smaller than 3 times the standard error of the estimate (SEE).

In [ ]:
# Extract distances and arrival times
[dis_obs,time_obs] = ffd.data_dis_time(ev_limit_dis_phase,st_info_obs)
# Linear regression
(slope,intercept,SEE) = ffd.linear_regression(dis_obs,time_obs)
up      =  3*SEE    # upper bound
down    = -3*SEE    # lower bound
print("The (slope,intercept,SEE) of original data is (%6.3f,%6.3f,%6.3f)"%(slope,intercept,SEE))

# See the English README for details.
dis_min = 0 
dis_max = 110
ev_remove_outlier_3SEE = ffd.fig_data_plot_remove_outliers(ev_limit_dis_phase,st_info_obs,slope,intercept,up,down,dis_min,dis_max, fname = "figs/step2_data_remove_outlier_3SEE.png")

# Evaluate the regression after removing outliers (optional)
(dis_obs,time_obs) = ffd.data_dis_time(ev_remove_outlier_3SEE,st_info_obs)
(slope_2,intercept_2,SEE_2) = ffd.linear_regression(dis_obs,time_obs)
print("The (slope,intercept,SEE) of filtered data is (%6.3f,%6.3f,%6.3f)"%(slope_2,intercept_2,SEE_2))

# 5. Output the Processed Data

In [ ]:
# Write data to the target directory
import os

# Specify the data directory
out_path = "output_data"
os.makedirs(out_path,exist_ok=True)

# Save as a TomoATT-format data file
out_fname = "%s/step2_src_rec_remove_outlier.dat"%(out_path)
ffd.write_src_rec_file(out_fname,ev_remove_outlier_3SEE,st_info_obs)